Connected to .venv (Python 3.12.8)

 # `aidweather` — Practical Guide

 **Author:** Cleverson Matiolli, PhD

 ---

 `aidweather` retrieves daily or hourly weather data from
 [NASA POWER](https://power.larc.nasa.gov/) and caches it locally so repeated
 requests don't hit the network again.

 This notebook walks through every public component of the package:

 | Section | What we do |
 |---------|-------------------|
 | 1 · Configuration | Inspect API endpoints and available weather parameters |
 | 2 · Coordinates | Parse, validate and convert geographic coordinates |
 | 3 · Fetching data | Download weather data for a single location |
 | 4 · Caching | See how the cache speeds up repeated queries |
 | 5 · Multi-point | Fetch data for several locations in parallel |
 | 6 · Data cleanup | Standardise the date column in any DataFrame |

In [1]:
import time
import pandas as pd
from pathlib import Path

from aidweather import (
    PowerClient,
    GeoCoordinate,
    normalize_coord_input,
    cfg,
    get_config,
    ensure_date_column,
)

print(f"aidweather ready.")

aidweather ready.


 ---
 ## 1 · Configuration

 `cfg` is a module-level singleton that reads `assets/config.json` once and
 exposes helpers for API URLs and parameter metadata.
 `get_config()` returns the same object — both refer to the same instance.

In [3]:
config = get_config()

# API base URLs
print("Daily point URL  :", config.get_url("daily", "point"))
print("Hourly point URL :", config.get_url("hourly", "point"))

Daily point URL  : https://power.larc.nasa.gov/api/temporal/daily/point
Hourly point URL : https://power.larc.nasa.gov/api/temporal/hourly/point


 ### Available parameter groups

 Parameters are grouped in `config.json`.
 The `"default"` group is a curated set of variables useful for most agronomic work.

In [4]:
default_params = config.params("default")
print(f"Default group — {len(default_params)} parameters:\n")
for code, label in default_params.items():
    print(f"  {code:<22} {label}")

Default group — 7 parameters:

  T2M                    Temperature at 2 m (°C)
  T2M_RANGE              Temperature Range at 2 m (°C)
  T2MDEW                 Dew Point Temperature at 2 m (°C)
  RH2M                   Relative Humidity at 2 m (%)
  PRECTOTCORR            Corrected Total Precipitation (mm/day)
  ALLSKY_SFC_PAR_TOT     All Sky Photosynthetically Active Radiation (MJ/m²/day)
  WS10M                  Wind Speed at 10 m (m/s)


 The `"all"` group is the full set of included in `aidweather` config variables.

In [7]:
default_params = config.params("all")
print(f"Default group — {len(default_params)} parameters:\n")
for code, label in default_params.items():
    print(f"  {code:<22} {label}")

Default group — 15 parameters:

  T2M                    Temperature at 2 m (°C)
  T2M_MAX                Maximum Temperature at 2 m (°C)
  T2M_MIN                Minimum Temperature at 2 m (°C)
  T2M_RANGE              Temperature Range at 2 m (°C)
  T2MWET                 Wet Bulb Temperature at 2 m (°C)
  T2MDEW                 Dew Point Temperature at 2 m (°C)
  TS                     Surface Skin Temperature (°C)
  RH2M                   Relative Humidity at 2 m (%)
  PRECTOTCORR            Corrected Total Precipitation (mm/day)
  ALLSKY_SFC_SW_DWN      All Sky Surface Shortwave Downward Radiation (kWh/m²/day)
  ALLSKY_SFC_PAR_TOT     All Sky Photosynthetically Active Radiation (MJ/m²/day)
  WS10M                  Wind Speed at 10 m (m/s)
  WS10M_MAX              Maximum Wind Speed at 10 m (m/s)
  WD10M                  Wind Direction at 10 m (degrees)
  PS                     Surface Pressure (kPa)


 ### Parameter descriptions

 Each parameter has a longer description stored in the config.

In [8]:
descriptions = config.param_descriptions()
for code in ["T2M", "PRECTOTCORR", "ALLSKY_SFC_PAR_TOT", "T2MDEW"]:
    print(f"[{code}]\n  {descriptions.get(code, 'n/a')}\n")

[T2M]
  Average Air Temperature at 2 m (°C): Modeled daily average air temperature. Data is obtained from the NASA MERRA-2 assimilation model (1981-Present, ~0.5°x0.625° resolution). In agriculture, T2M is critical for calculating Growing Degree Days (GDD) and modeling crop developmental stages and growth rates. Used in machine learning models for yield prediction and thermal stress assessment.

[PRECTOTCORR]
  Corrected Total Precipitation (mm/day): Total liquid and frozen water precipitation, adjusted for known biases. Data is obtained from the NASA MERRA-2 model (1981/1997-Present, ~0.5°x0.625° resolution). Essential for water balance models, irrigation scheduling, and assessing drought/flood risk. It's a primary input feature for deep learning models predicting crop yield.

[ALLSKY_SFC_PAR_TOT]
  All Sky Photosynthetically Active Radiation (MJ/m²/day): The total energy flux (400-700 nm) used by plants for photosynthesis. Data is derived from NASA satellite observations (1984-Presen

 ---
 ## 2 · Coordinates

 `GeoCoordinate` accepts coordinates in **common format** and converts
 between them.  It also validates ranges before you ever touch the API.

 Supported input styles:
 - Decimal degrees as a float: `-23.55`
 - Decimal degrees as a string: `"-23.55°"`
 - Degrees and decimal minutes: `"23° 33.0' S"`
 - Degrees, minutes, seconds: `"23° 33' 0\" S"`

In [9]:
# From two strings (any mix of formats)
c1 = GeoCoordinate.from_strings("23° 33.0' S", "46° 37' 48\" W")

# From two floats (south/west are negative)
c2 = GeoCoordinate.from_decimal(-23.55, -46.63)

for i, c in enumerate([c1, c2], 1):
    print(f"Coord {i}:  lat={c.lat:.5f}  lon={c.lon:.5f}")

Coord 1:  lat=-23.55000  lon=-46.63000
Coord 2:  lat=-23.55000  lon=-46.63000


 ### Mixed-format input via `normalize_coord_input`

 If you have a tuple where each element could be a float or a string,
 `normalize_coord_input` handles it in one call.

In [10]:
coord = normalize_coord_input((-23.55, "46°37'48.0\" W"))
print("Normalised:", coord)

Normalised: GeoCoordinate(lat=-23.55, lon=-46.63)


 ### Export to formatted strings

In [ ]:
print("DD  :", coord.to_dd_str(lat_precision=5))
print("DDM :", coord.to_ddm_str(minute_precision=3))
print("DMS :", coord.to_dms_str(second_precision=2))

 ### Validation

 Invalid coordinates raise a `ValueError` immediately — before any API call is made.

In [ ]:
try:
    GeoCoordinate.from_decimal(95.0, -46.63)   # latitude > 90 is invalid
except ValueError as e:
    print("Caught:", e)

 ---
 ## 3 · Fetching weather data

 `PowerClient` wraps the NASA POWER API.
 Initialise it once and reuse it across queries.

 ```python
 client = PowerClient(temporal_api="daily")   # or "hourly"
 ```

 ### `get_point_data` — single location

 Returns a **pandas DataFrame** indexed by date, one column per parameter.

In [ ]:
client = PowerClient(temporal_api="daily")

lat, lon = -23.31, -51.16          # Londrina, PR, Brazil
start, end = "2023-01-01", "2023-01-15"
params = ["T2M", "PRECTOTCORR", "ALLSKY_SFC_PAR_TOT"]

df = client.get_point_data(lat=lat, lon=lon, start=start, end=end, params=params)

print(df.shape)   # (days, parameters)
print(df.head())

 ### `summarize` — quick data overview

 Prints a formatted table with row counts, date range and per-column statistics.

In [ ]:
client.summarize(df)